# Solution to Date and Time Manipulation

## Load Libraries

In [87]:
import pandas as pd
import polars as pl
import numpy as np

## Generate Sample Data

I use snippets from Copilot to generate random sequences of dates in Pandas first, then create a Polars copy of the result for the exercises.

In [88]:
start = pd.to_datetime("2020-01-01")
end = pd.to_datetime("2023-12-31")

dates_df_pd = pd.DataFrame({
    "start_time": pd.to_datetime(
    np.random.randint(start.value, end.value, size=20, dtype=np.int64)
).strftime("%Y-%m-%d %H:%M:%S"),
"end_time": pd.to_datetime(
    np.random.randint(start.value, end.value, size=20, dtype=np.int64)
).strftime("%Y-%m-%d %H:%M:%S")
})

dates_df_pl = pl.from_pandas(dates_df_pd)

# Pandas Solution

## Conversion `across`

I use the `apply` method to convert each column individually to `datetime` type. This has the same effect as `across` in `dplyr`. The definition of `cols` is unnecessary in this case because I am applying the transformation to all columns in the data frame, but I leave it in so that we can use this as a template later on if we want to select for a subset of columns based on data type and perform a transformation on those columns only (replacing the current data type of `'O'` (object) with our desired data type).

## New Columns `across`

I use the `assign` method to create two new columns: one for the day of the week of each of `start_time` and `end_time`.

## Syntax Note

If we want to use method chaining (as below), note that the variable has to be contained in a set of parenthesis or there will be an 'unexpected indent' error. 

In [89]:
cols = dates_df_pd.select_dtypes('O').columns
dates_df_pd[cols] = (
    dates_df_pd[cols]
        .apply(pd.to_datetime)
)

dates_df_pd['duration'] = dates_df_pd['end_time'] - dates_df_pd['start_time']

date_cols = dates_df_pd.select_dtypes('datetime')

dates_df_pd = (
    dates_df_pd
        .assign(**{
            f"{c}_weekday": (dates_df_pd[c].dt.day_name())
            for c in date_cols
        })
)

# Polars Solution

## `across` Equivalents

Note that I had to use separate `with_columns` methods chained together in order to acomplish the conversion and calculation steps. I originally tried including these in the same `with_columns` method using a list, but that caused errors because the columns had not been altered in the correct order for the operations (I think). For adding new columns, Copilot originally told me to introduce a `map_elements` method into the last `with_columns` method but that caused errors. It was unnecessary for this application because there are built in functions that achieve the same result, and it apparently slows calculations down and should be used as a last result.

In [90]:
dates_df_pl = (
    dates_df_pl
        .with_columns(
            pl.all().str.to_datetime()
        )
        .with_columns(
            duration = pl.col('end_time') - pl.col('start_time')
        )
        .with_columns(
            pl.col(r"^.*_time$").dt.strftime("%A").name.suffix("_weekday")
        )
)

# Bonus Question Answer

In my opinion, the Polars solution is much more R-like, with a single chain of methods. 